In [5]:
import json
import os
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
from IPython.display import display, HTML

# Caminhos dos arquivos
CAMINHO_ORIGINAL = "../assets/Crossroads of Twilight.epub"
ARQUIVO_JSON = "estrutura_livro_traduzido.json"
CAMINHO_SAIDA_TESTE = "livro_remontado_teste.epub"

print("✓ Ambiente configurado.")

✓ Ambiente configurado.


In [6]:
# 1. Carrega o livro original e os dados do JSON
livro_original = epub.read_epub(CAMINHO_ORIGINAL)

with open(ARQUIVO_JSON, 'r', encoding='utf-8') as f:
    dados_json = json.load(f)

# 2. Cria o novo objeto do livro
novo_livro = epub.EpubBook()

# Configura os metadados
novo_livro.set_title(dados_json.get("titulo_arquivo", "Livro Traduzido").replace(".epub", ""))
novo_livro.set_language('pt') 

# Copia tudo o que não for texto do livro original
for item in livro_original.get_items():
    if item.get_type() in [ebooklib.ITEM_IMAGE, ebooklib.ITEM_STYLE, ebooklib.ITEM_FONT]:
        novo_livro.add_item(item)

print(f"✓ Metadados e mídias copiados com sucesso!")

✓ Metadados e mídias copiados com sucesso!


In [7]:
import re  # IMPORTANTE: Importamos o módulo de expressões regulares

lista_novos_capitulos = []

# Varre os capítulos mapeados no JSON
for cap_json in dados_json["capitulos"]:
    nome_interno = cap_json["nome_arquivo_interno"]
    titulo_capitulo = cap_json["titulo_capitulo"]
    
    # Estrutura básica XHTML
    html_novo = '<?xml version="1.0" encoding="utf-8"?>\n'
    html_novo += '<html xmlns="http://www.w3.org/1999/xhtml">\n<head>\n'
    html_novo += f'<title>{titulo_capitulo}</title>\n'
    
    # CSS AJUSTADO: Adicionado 'white-space: pre-wrap;' para não perder os espaçamentos e quebras de linha
    html_novo += '<style>\n'
    html_novo += '  .ref-trad { border-bottom: 1px dashed #5c93c4; cursor: help; white-space: pre-wrap; }\n'
    html_novo += '  em { font-style: italic; }\n'
    html_novo += '</style>\n'
    html_novo += '</head>\n<body>\n'
    
    # Varre o conteúdo do capítulo específico
    for elemento in cap_json["conteudo"]:
        if elemento["tipo"] == "texto":
            tag = elemento["tag_original"]
            
            texto_final = elemento["traduzido"] if elemento["traduzido"] else elemento["original"]
            texto_original = elemento["original"]
            
            # 1. LIMPEZA DE TAGS RESIDUAIS: Remove os blocos de ```html ou ``` que escaparam no texto
            texto_final = texto_final.replace("```html", "").replace("```", "")
            texto_original = texto_original.replace("```html", "").replace("```", "")
            
            # 2. LIMPEZA DE ASPAS, CRASES E ESPAÇOS SOBRANDO NAS EXTREMIDADES
            # Remove crases, aspas simples/duplas e quebras de linha que estejam grudadas nas pontas do texto
            texto_final = texto_final.strip().strip("`").strip("'").strip('"').strip()
            texto_original = texto_original.strip().strip("`").strip("'").strip('"').strip()
            
            # 3. MANUTENÇÃO DE QUEBRAS DE LINHA: Transforma quebras de linha normais em quebras de linha HTML (<br />)
            # Isso impede que os textos de copyright, créditos e endereços fiquem todos grudados
            texto_final = texto_final.replace("\n", "<br />")
            texto_original = texto_original.replace("\n", "<br />")
            
            # 4. CONVERSÃO DE ASTERISCOS PARA ITÁLICO (EM)
            texto_final = re.sub(r'\*(.*?)\*', r'<em>\1</em>', texto_final)
            
            # Tratamento para aspas não quebrarem o atributo 'title' do HTML
            texto_original_escapado = texto_original.replace('"', '&quot;') 
            # Também limpamos os asteriscos do texto original que vai pro Hover
            texto_original_escapado = re.sub(r'\*(.*?)\*', r'\1', texto_original_escapado)
            
            # Monta a estrutura de Hover usando a classe CSS que criamos acima
            frase_com_hover = f'<span class="ref-trad" title="ORIGINAL: {texto_original_escapado}">{texto_final}</span>'
            
            # Reconstrói a tag (ex: <p>, <h1>) com o conteúdo dentro
            html_novo += f'<{tag}>{frase_com_hover}</{tag}>\n'
            
        elif elemento["tipo"] == "imagem":
            src = elemento["src_original"]
            html_novo += f'<img src="{src}" />\n'
            
    html_novo += '</body>\n</html>'
    
    # Cria o objeto EpubHtml da biblioteca
    novo_capitulo = epub.EpubHtml(
        title=titulo_capitulo,
        file_name=nome_interno,
        content=html_novo.encode('utf-8')
    )
    
    # Adiciona o capítulo ao livro e monitora na lista
    novo_livro.add_item(novo_capitulo)
    lista_novos_capitulos.append(novo_capitulo)

print(f"✓ Todos os {len(lista_novos_capitulos)} capítulos foram processados com limpezas de formatação aplicadas!")

✓ Todos os 48 capítulos foram processados com limpezas de formatação aplicadas!


In [8]:
# 1. Cria os objetos de navegação do EPUB de forma correta
nav_item = epub.EpubNav()
ncx_item = epub.EpubNcx()

# 2. Adiciona os objetos de navegação ao livro (PASSO CRÍTICO QUE FALTAVA)
novo_livro.add_item(nav_item)
novo_livro.add_item(ncx_item)

# 3. Define o Spine (Ordem de leitura). O 'nav' precisa estar aqui dentro
novo_livro.spine = ['nav'] + lista_novos_capitulos

# 4. Define o TOC (Sumário lateral que os leitores usam)
novo_livro.toc = tuple(lista_novos_capitulos)

# 5. Grava o arquivo final no disco
epub.write_epub(CAMINHO_SAIDA_TESTE, novo_livro)

print(f"🎉 SUCESSO! Livro completo gerado em: {os.path.abspath(CAMINHO_SAIDA_TESTE)}")
print("\n--- PRÉVIA INTERATIVA DA PRIMEIRA FRASE DO LIVRO NO JUPYTER ---")

# Mostra uma prévia interativa na tela do seu Notebook para você testar o Hover agora mesmo
if lista_novos_capitulos:
    html_preview = lista_novos_capitulos[0].get_content().decode('utf-8')
    display(HTML(html_preview))

🎉 SUCESSO! Livro completo gerado em: d:\traducao\livro10\livro_remontado_teste.epub

--- PRÉVIA INTERATIVA DA PRIMEIRA FRASE DO LIVRO NO JUPYTER ---
